## **LangChain** - _Embeddings_

> Un **Embedding** es una técnica que traduce lenguaje humano (palabras, frases o documentos enteros) en secuencias de números (vectores de alta dimensionalidad) que las máquinas pueden procesar. Su mayor cualidad es que capturan el significado semántico: textos con conceptos similares generarán vectores que matemáticamente apuntan en la misma dirección o están muy cerca en un espacio multidimensional.

### Características principales:
*   **Densidad:** A diferencia de métodos clásicos (como Bag of Words), los vectores son densos (pocos ceros).
*   **Espacio Vectorial:** Los embeddings sitúan las palabras en un espacio multidimensional. Palabras con significados similares terminan en coordenadas cercanas dentro de ese espacio.
*   **Relaciones Matemáticas:** Permiten realizar operaciones matemáticas. El ejemplo clásico es:
    $$ Rey - Hombre + Mujer \approx Reina $$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.utils.math import cosine_similarity
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from dotenv import load_dotenv

load_dotenv()

### 01. Embeddings

In [ ]:
EMBEDDING_MODEL = "gemini-embedding-2"
OUTPUT_DIMENSIONALITY = 3072 # 128 a 3072

embeddings_model = GoogleGenerativeAIEmbeddings(model = EMBEDDING_MODEL, output_dimensionality = OUTPUT_DIMENSIONALITY)

texto = """Un Embedding es una técnica que traduce lenguaje humano (palabras, frases o documentos enteros) en secuencias de números 
           (vectores de alta dimensionalidad) que las máquinas pueden procesar. Su mayor cualidad es que capturan el significado semántico:
           textos con conceptos similares generarán vectores que matemáticamente apuntan en la misma dirección o están muy cerca en un espacio multidimensional."""

# Generar el embedding
embedding = embeddings_model.embed_query(text = texto)

print(f"Dimensión del vector: {len(embedding)}")

In [ ]:
embedding

In [ ]:
# Generar embeddings a partir de una lista de strings

oraciones = ["Me encanta programar en Python.",
             "El desarrollo de software con Python es genial.",
             "Hoy hace un día soleado en Madrid.",
             "El clima está despejado y brillante hoy.",
             "El papa está en Barcelona."]

embeddings = embeddings_model.embed_documents(texts = oraciones)

print(f"Documentos procesados: {len(embeddings)}")
print(f"Dimensión del primer documento: {len(embeddings[0])}")

### 02. Similitud Semántica

La similitud semántica es una métrica empleada en lingüística y en inteligencia artificial que mide el grado en que dos palabras, frases o conceptos comparten un mismo significado o intención, independientemente de las palabras literales que utilicen.

A diferencia de la coincidencia exacta de texto (que busca palabras iguales), la similitud semántica comprende el contexto. Permite a las máquinas entender que frases diferentes tienen un propósito equivalente.

- Para calcular la similitud semántica se utiliza principalmente la similitud de coseno:
    - Esta función mide el coseno del ángulo formado por dos vectores en un espacio multidimensional. Cuando los textos se transforman en vectores, esta fórmula evalúa qué tan alineados están sus significados.
    
$$
\text{Similitud de Coseno} = \frac{\mathbf{A}\cdot \mathbf{B}}{\|\mathbf{A}\|\|\mathbf{B}\|}=\frac{\sum _{i=1}^{n}A_{i}B_{i}}{\sqrt{\sum _{i=1}^{n}A_{i}^{2}}\sqrt{\sum _{i=1}^{n}B_{i}^{2}}}\
$$


- La razón fundamental por la que se elige esta función sobre otras (como la distancia Euclidiana) es que ignora la longitud de los textos y se enfoca estrictamente en la dirección del significado:

    - **Independencia del tamaño**: Si un documento repite la palabra "computadora" 10 veces y otro texto corto la menciona solo una vez, sus vectores apuntarán en la misma dirección, pero uno será mucho más largo que el otro. El coseno ignora esa diferencia de tamaño.
    - **Escala normalizada**: El resultado siempre oscila en un rango fijo entre -1 y 1 (o entre 0 y 1) en la mayoría de los sistemas de embeddings. Esto facilita establecer umbrales automáticos (por ejemplo, decidir que cualquier resultado mayor a 0.85 es un sinónimo).
    - **Interpretación del resultados:**
        - Resultado igual a 1: Los vectores son idénticos y apuntan exactamente en la misma dirección. Significa máxima similitud semántica.
        - Resultado igual a 0: Los vectores son perpendiculares u ortogonales. Significa que no comparten ninguna relación semántica.
        - Resultado igual a -1: Los vectores apuntan en direcciones opuestas. Significa que son conceptos opuestos.

In [ ]:
# Calculamos la similitud del coseno entre todas las oraciones
similitudes = cosine_similarity(X = embeddings, Y = embeddings)

pd.DataFrame(similitudes)

In [ ]:
oraciones

In [ ]:
# Oración 0 y la oración 1
print(f"Oración 1: '{oraciones[0]}'")
print(f"Oración 2: '{oraciones[1]}'")
print(f"Puntuación de similitud: {similitudes[0][1]:.4f}")

# Oración 0 y la oración 2
print(f"\nOración 1: '{oraciones[0]}'")
print(f"Oración 2: '{oraciones[2]}'")
print(f"Puntuación de similitud: {similitudes[0][2]:.4f}")

### 03. Búsqueda Semántica (Motor de Búsqueda de IA)

Buscar información por su significado y no por palabras clave exactas.

In [ ]:
corpus = ["Un perro está persiguiendo una pelota.",
          "Un hombre está comiendo una manzana.",
          "La inteligencia artificial está transformando el mundo.",
          "Un cachorro juega con su juguete en el parque."]

consulta = "Un animal pequeño divirtiéndose"

# Creamos el VectorStore directamente con los textos y el modelo
vectorstore = FAISS.from_texts(texts = corpus,
                               embedding = embeddings_model,
                               distance_strategy = DistanceStrategy.COSINE)

# Realizamos la búsqueda semántica
resultados = vectorstore.similarity_search_with_score(query = consulta, k = 4)

print(f"Consulta: {consulta}\n")
for doc, score in resultados:
    print(f"Documento encontrado: '{doc.page_content}' (Distancia: {score:.4f})")

# El resultado muestra la distancia entre embeddings, los valores más pequeños son los que tienen mayor parecido semántico

### 04. Clustering (Agrupación de Textos)

Podemos usar embeddings junto con algoritmos tradicionales de Machine Learning para agrupar textos similares automáticamente, ideal para organizar tickets de soporte o reseñas de clientes.

In [ ]:
textos_variados = ["La nueva tarjeta gráfica es rapidísima.",
                   "El pastel de chocolate estaba delicioso.",
                   "Mi monitor parpadea y no enciende.",
                   "Necesito la receta de la tarta de queso.",
                   "El procesador se calienta demasiado.",
                   "Es el momento de pedir una tarta red velvet.",
                   "Las tarjetas graficas están muy costosas."]

embeddings = embeddings_model.embed_documents(texts = textos_variados)

# Queremos agruparlos en 2 categorías (Tecnología vs Comida)
num_clusters = 2
kmeans = KMeans(n_clusters = num_clusters, random_state = 42)
kmeans.fit(embeddings)

grupos = kmeans.labels_

print("Agrupación automática de textos:")

for i in range(num_clusters):

    print(f"--- Clúster {i + 1} ---")

    for texto, grupo in zip(textos_variados, grupos):

        if grupo == i:

            print(f"- {texto}")

### 05. Visualización 2D de Embeddings

In [ ]:
# Reducción de Dimensionalidad
pca = PCA(n_components = 2)
embeddings_2d = pca.fit_transform(embeddings)

df = pd.DataFrame(data = embeddings_2d, columns = ["pca1", "pca2"])
df["grupos"] = grupos
df["texto"] = textos_variados

# Scatter Plot
px.scatter(data_frame = df, x = "pca1", y = "pca2", color = "grupos", hover_name = "texto")